# Notebook 02: NCCL Collectives

## Why This Matters

Distributed training requires GPUs to communicate: share gradients, broadcast parameters, scatter data. The efficiency of this communication determines whether adding more GPUs actually speeds up training or just adds overhead. NCCL (NVIDIA Collective Communications Library) implements these operations optimally. Understanding ring all-reduce explains why 8 GPUs can achieve near-linear scaling for training but not for inference.


In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.distributed as dist
import numpy as np
import matplotlib.pyplot as plt
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")
print(f"Available GPUs: {torch.cuda.device_count()}")
print("Ready.")

## 1. Why Distributed Training Needs Collectives

In **data parallel training**, each GPU:
1. Gets a different mini-batch shard
2. Computes the forward + backward pass independently
3. Ends up with **different gradients** (because different data)

To take the correct gradient step, all GPUs must agree on the **average gradient** across all shards. This requires a collective communication operation.

The naive approach: each GPU sends its gradients to a central server, which averages and broadcasts back. Cost: O(P) communication per GPU, where P = parameter count. For a 7B model: 7e9 * 4 bytes = 28 GB. With 8 GPUs, the parameter server sees 224 GB inbound.

**Ring all-reduce** is dramatically better.


## 2. Ring All-Reduce: The Bandwidth-Optimal Algorithm

Ring all-reduce organizes N GPUs in a ring. It runs in two phases:

**Phase 1 (reduce-scatter)**: Each GPU splits its tensor into N chunks. In N-1 steps, each GPU sends its chunk to the next GPU (which adds it to its own). After N-1 steps, each GPU has one chunk that is the sum of all GPUs' versions of that chunk.

**Phase 2 (all-gather)**: In N-1 more steps, each GPU broadcasts its reduced chunk around the ring until every GPU has all chunks.

**Total data sent per GPU**: 2 * (N-1)/N * P bytes ≈ 2P bytes (independent of N!)

Compare to parameter server: O(N * P) total -- scales with GPU count. Ring all-reduce: O(P) total -- scales only with data size.

This is why adding more GPUs does not increase communication volume per GPU in ring all-reduce.


In [ ]:
# Simulate ring all-reduce algorithm
def simulate_ring_allreduce(tensors):
    N = len(tensors)
    chunks = [t.clone() for t in tensors]
    
    # Phase 1: Reduce-scatter
    # After this, chunks[i] holds the sum of rank i's shard
    # (we simplify: treat each tensor as one chunk)
    # Each rank sends to (rank+1) % N, receives from (rank-1+N) % N
    
    # Phase 2 conceptual: 
    # after reduce-scatter: each rank has a fully-reduced shard
    # after all-gather: each rank has all fully-reduced shards
    
    # Result: each rank should have sum of all
    true_sum = sum(t.clone() for t in tensors)
    return true_sum

# Demonstrate with small tensors
N_gpus = 4
param_size = 8  # 8 parameters for illustration

# Simulate: each GPU has different gradients
torch.manual_seed(42)
gpu_grads = [torch.randn(param_size) for _ in range(N_gpus)]

print('Each GPU has its own gradients:')
for i, g in enumerate(gpu_grads):
    print(f'  GPU {i}: {g.numpy().round(2)}')

true_mean = torch.stack(gpu_grads).mean(dim=0)
print(f'\nTrue mean gradient: {true_mean.numpy().round(2)}')
print(f'\nAfter ring all-reduce, all GPUs should have this mean.')

# Communication cost analysis
def comm_cost_ring(n_params, n_gpus, bytes_per_param=4):
    # Each GPU sends/receives 2 * (N-1)/N * P bytes
    return 2 * (n_gpus - 1) / n_gpus * n_params * bytes_per_param

def comm_cost_param_server(n_params, n_gpus, bytes_per_param=4):
    # Each GPU sends P bytes to server, server sends P bytes back
    return 2 * n_params * bytes_per_param

n_params = 7e9  # 7B model
print('\nCommunication cost per GPU (7B parameter model):')
print(f'GPU count | Ring all-reduce | Param server')
print('-' * 45)
for n in [2, 4, 8, 16, 32, 64]:
    ring_gb = comm_cost_ring(n_params, n) / 1e9
    ps_gb = comm_cost_param_server(n_params, n) / 1e9
    print(f'{n:9d} | {ring_gb:15.1f} GB | {ps_gb:.1f} GB')
print('\nRing all-reduce cost is constant per GPU!')
print('Parameter server cost is constant per GPU too, but the SERVER sees N*P GB.')

## 3. All the Collectives

| Collective | What it does | Use case |
|------------|-------------|---------|
| **All-reduce** | Every rank contributes; every rank gets the sum/mean | Gradient averaging in DDP |
| **Reduce-scatter** | Every rank contributes; each rank gets one chunk of the result | FSDP gradient sharding |
| **All-gather** | Each rank has one chunk; every rank ends up with all chunks | FSDP parameter gathering |
| **Broadcast** | One rank sends to all | Broadcasting initial model weights |
| **Reduce** | Every rank contributes; ONE rank gets the result | Loss aggregation for logging |
| **Scatter** | One rank distributes different chunks to each rank | Distributing data shards |
| **Gather** | Each rank sends to one rank | Collecting results |
| **Barrier** | Synchronization -- all wait until all reach barrier | Ensuring all ranks finish before proceeding |


In [ ]:
# Visualize collective operations
fig, axes = plt.subplots(2, 4, figsize=(16, 6))

def draw_collective(ax, title, before, after, arrow_pattern):
    N = len(before)
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlim(-0.5, 3.5)
    ax.set_ylim(-0.5, N - 0.5)
    ax.axis('off')
    
    for i in range(N):
        ax.text(0.5, i, f'R{i}: {before[i]}', ha='center', va='center',
               bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.7),
               fontsize=8)
        ax.text(2.5, i, f'R{i}: {after[i]}', ha='center', va='center',
               bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7),
               fontsize=8)
    
    ax.annotate('', xy=(1.7, (N-1)/2), xytext=(1.3, (N-1)/2),
               arrowprops=dict(arrowstyle='->', color='black', lw=2))
    ax.text(1.5, (N-1)/2 + 0.3, arrow_pattern, ha='center', fontsize=7)

N = 4
rank_data = ['g0', 'g1', 'g2', 'g3']
all_sum = 'g0+g1+g2+g3'

draw_collective(axes[0,0], 'All-Reduce\n(gradient averaging)',
               rank_data, [all_sum]*4, 'ring comm')

draw_collective(axes[0,1], 'Reduce-Scatter\n(FSDP phase 1)',
               rank_data, ['s0', 's1', 's2', 's3'], 'shards')

draw_collective(axes[0,2], 'All-Gather\n(FSDP phase 2)',
               ['s0', 's1', 's2', 's3'], [all_sum]*4, 'broadcast')

draw_collective(axes[0,3], 'Broadcast\n(weight init)',
               ['W', '-', '-', '-'], ['W', 'W', 'W', 'W'], '1->all')

draw_collective(axes[1,0], 'Reduce\n(loss logging)',
               rank_data, [all_sum, '-', '-', '-'], 'N->1')

draw_collective(axes[1,1], 'Scatter\n(data dist)',
               ['d0d1d2d3', '-', '-', '-'], ['d0', 'd1', 'd2', 'd3'], '1->N')

draw_collective(axes[1,2], 'Gather\n(result collect)',
               ['d0', 'd1', 'd2', 'd3'], ['d0d1d2d3', '-', '-', '-'], 'N->1')

draw_collective(axes[1,3], 'Barrier\n(sync point)',
               ['wait', 'wait', 'wait', 'wait'], ['go', 'go', 'go', 'go'], 'all wait')

for ax in axes.flat:
    ax.set_aspect('auto')

plt.suptitle('NCCL Collective Operations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('nccl_collectives.png', dpi=100)
plt.show()

## 4. NCCL vs Gloo vs MPI

| Library | Transport | Best for | Used in |
|---------|-----------|---------|---------|
| **NCCL** | NVLink, PCIe, InfiniBand | GPU-to-GPU (NVIDIA) | PyTorch DDP default on GPU |
| **Gloo** | TCP, shared memory | CPU, debugging | PyTorch DDP fallback on CPU |
| **MPI** | InfiniBand, UCX | HPC clusters, multi-node | Academic HPC, some production |
| **RCCL** | AMD equivalent of NCCL | AMD ROCm GPUs | AMD GPU clusters |

**Latency vs bandwidth tradeoff**:
- Small messages (< 32 KB): **latency-bound** -- use tree algorithms (fewer hops)
- Large messages (> 1 MB): **bandwidth-bound** -- use ring algorithms (optimal bandwidth)

NCCL automatically selects the best algorithm based on message size and topology.


## Summary

### Key Concepts

| Collective | Use case | Cost |
|------------|---------|------|
| All-reduce (ring) | DDP gradient averaging | 2*(N-1)/N * P bytes per GPU |
| Reduce-scatter | FSDP gradient sharding | P/N bytes received per GPU |
| All-gather | FSDP parameter reconstruction | P bytes received per GPU |
| Broadcast | Initial model weight sync | P bytes per non-root GPU |

**The golden rule**: ring all-reduce costs 2P bytes per GPU regardless of how many GPUs you have. This is why DDP scales linearly -- communication cost per GPU stays constant as you add more GPUs.

**Next**: `03_ddp_deep_dive.ipynb` -- PyTorch DDP internals: bucketing, overlap, and debugging.
